In [2]:

import re, subprocess
from collections import defaultdict
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

ICONS = {
    'nucleare':'☢️','sanzioni':'🔒','opinione':'📣','defcon':'🚨',
    'risorse_iran':'💵','forze_militari_iran':'⚔️','tecnologia_nucleare_iran':'🔬','stabilita_iran':'⚖️',
    'risorse_coalizione':'💵','influenza_diplomatica_coalizione':'🤝','tecnologia_avanzata_coalizione':'💻',
    'supporto_pubblico_coalizione':'📢','stabilita_coalizione':'⚖️',
    'risorse_russia':'💵','influenza_militare_russia':'🎖️','veto_onu_russia':'🏛️',
    'stabilita_economica_russia':'📊','stabilita_russia':'⚖️',
    'risorse_cina':'💵','influenza_commerciale_cina':'🏪','cyber_warfare_cina':'🖥️',
    'stabilita_rotte_cina':'🚢','stabilita_cina':'⚖️',
    'risorse_europa':'💵','influenza_diplomatica_europa':'🕊️','aiuti_umanitari_europa':'❤️',
    'coesione_ue_europa':'🌐','stabilita_europa':'⚖️'
}

GLOBAL_TRACKS = ['nucleare','sanzioni','opinione','defcon']
ALL_TRACKS = list(DEFAULT.keys())

def eval_ternary(expr, v):
    expr = expr.strip().rstrip(',')
    def replace_ternary(e):
        pattern = re.compile(r'([^?:]+)\?([^?:]+):([^?:]+)')
        while '?' in e:
            e = pattern.sub(lambda m: f"({m.group(2)} if {m.group(1)} else {m.group(3)})", e, count=1)
        return e
    try:
        return float(eval(replace_ternary(expr), {'v': v}))
    except:
        return None

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)',\s*description:'([^']*)'.*?effects:\{([^}]+)\}", re.DOTALL)

rows = []
for m in card_re.finditer(src):
    cid, cname, faz, ctype, op, deck, desc, eff_str = m.groups()
    row = {'id': cid, 'nome': cname, 'fazione': faz, 'tipo': ctype, 'op': int(op), 'mazzo': deck, 'descrizione': desc}
    for t in ALL_TRACKS:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            r = eval_ternary(fn.group(1), DEFAULT[t])
            if r is not None:
                delta = round(r - DEFAULT[t], 1)
                if t in GLOBAL_TRACKS:
                    delta = max(-3, min(3, delta))  # clip ±3 per globali
                row[t] = delta
            else:
                row[t] = 0
        else:
            row[t] = 0
    rows.append(row)

print(f"Carte caricate: {len(rows)}")

# Crea workbook
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Carte"

# Sezioni intestazioni
SECTIONS = {
    'Globali': (['nucleare','sanzioni','opinione','defcon'], 'FF1F3C6E'),    # blu scuro
    'Iran': (['risorse_iran','forze_militari_iran','tecnologia_nucleare_iran','stabilita_iran'], 'FF8B0000'),  # rosso scuro
    'Coalizione': (['risorse_coalizione','influenza_diplomatica_coalizione','tecnologia_avanzata_coalizione','supporto_pubblico_coalizione','stabilita_coalizione'], 'FF000080'),  # blu navy
    'Russia': (['risorse_russia','influenza_militare_russia','veto_onu_russia','stabilita_economica_russia','stabilita_russia'], 'FFB85C00'),  # arancio
    'Cina': (['risorse_cina','influenza_commerciale_cina','cyber_warfare_cina','stabilita_rotte_cina','stabilita_cina'], 'FF1B5E20'),  # verde scuro
    'Europa': (['risorse_europa','influenza_diplomatica_europa','aiuti_umanitari_europa','coesione_ue_europa','stabilita_europa'], 'FF4A148C'),  # viola
}

# Colonne base
base_cols = ['ID','Nome','Fazione','Tipo','OP','Mazzo','Descrizione']
track_cols = []
for sec, (tracks, _) in SECTIONS.items():
    track_cols.extend(tracks)

all_cols = base_cols + track_cols
ws.append(all_cols)

# Formattazione header
for col_idx, col_name in enumerate(all_cols, 1):
    cell = ws.cell(1, col_idx)
    cell.font = Font(bold=True, color='FFFFFFFF', size=10)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    
    # Colore sezione
    if col_name in base_cols:
        cell.fill = PatternFill('solid', fgColor='FF2C3E50')
    else:
        for sec, (tracks, color) in SECTIONS.items():
            if col_name in tracks:
                cell.fill = PatternFill('solid', fgColor=color)
                break
    
    # Label con icona
    icon = ICONS.get(col_name, '')
    if icon:
        cell.value = f"{icon}\n{col_name}"
    
ws.row_dimensions[1].height = 45

# Colori fazione per righe
FACTION_COLORS = {
    'Iran': 'FFFFF0F0', 'Coalizione': 'FFF0F4FF', 'Russia': 'FFFFF8F0',
    'Cina': 'FFF0FFF0', 'Europa': 'FFFDF0FF', 'Neutrale': 'FFF5F5F5'
}

# Fill colori delta
GREEN_FILL = PatternFill('solid', fgColor='FFD5F5D5')   # verde chiaro per +
RED_FILL   = PatternFill('solid', fgColor='FFFFD5D5')   # rosso chiaro per -
ZERO_FILL  = PatternFill('solid', fgColor='FFFFFFFF')

# Dati
for row in rows:
    data = [row['id'], row['nome'], row['fazione'], row['tipo'], row['op'], row['mazzo'], row['descrizione']]
    for t in track_cols:
        data.append(row.get(t, 0))
    
    ws.append(data)
    row_idx = ws.max_row
    faz_fill = PatternFill('solid', fgColor=FACTION_COLORS.get(row['fazione'], 'FFFFFFFF'))
    
    for col_idx in range(1, len(base_cols)+1):
        ws.cell(row_idx, col_idx).fill = faz_fill
        ws.cell(row_idx, col_idx).alignment = Alignment(wrap_text=True, vertical='top')
    
    for col_idx, t in enumerate(track_cols, len(base_cols)+1):
        val = row.get(t, 0)
        cell = ws.cell(row_idx, col_idx)
        cell.value = val if val != 0 else ''
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if val > 0:
            cell.fill = GREEN_FILL
            cell.font = Font(bold=True, color='FF1B5E20')
        elif val < 0:
            cell.fill = RED_FILL
            cell.font = Font(bold=True, color='FF8B0000')
        else:
            cell.fill = faz_fill

# Larghezze colonne
ws.column_dimensions['A'].width = 9
ws.column_dimensions['B'].width = 28
ws.column_dimensions['C'].width = 12
ws.column_dimensions['D'].width = 12
ws.column_dimensions['E'].width = 5
ws.column_dimensions['F'].width = 8
ws.column_dimensions['G'].width = 35
for col_idx in range(8, len(all_cols)+1):
    ws.column_dimensions[get_column_letter(col_idx)].width = 10

# Freeze + filtri
ws.freeze_panes = 'A2'
ws.auto_filter.ref = f"A1:{get_column_letter(len(all_cols))}{ws.max_row}"

wb.save('/workspace/linea_rossa_carte_completo.xlsx')
print(f"Excel salvato: {ws.max_row-1} carte, {len(all_cols)} colonne")


Carte caricate: 344


Excel salvato: 344 carte, 35 colonne
